In [1]:

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import shutil
import torch
from ultralytics import YOLO


In [2]:
print("PyTorch Version :", torch.__version__)
print("CUDA Version    :", torch.version.cuda)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. Training will use CPU.")

PyTorch Version : 2.13.0+cu132
CUDA Version    : 13.2
CUDA Available  : True
GPU             : NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
dataset_path = Path("Traffic Violations Dataset")

train_path = dataset_path / "train"
val_path = dataset_path / "validation"
test_path = dataset_path / "test"


# Check dataset folders
if not dataset_path.exists():
    raise FileNotFoundError(
        "helmet_dataset folder was not found."
    )

if not train_path.exists():
    raise FileNotFoundError(
        "Training folder not found."
    )

if not val_path.exists():
    raise FileNotFoundError(
        "Validation folder not found."
    )


print("\nDataset Path :", dataset_path.resolve())
print("Train Path   :", train_path.resolve())
print("Validation   :", val_path.resolve())
print("Test Path    :", test_path.resolve())



Dataset Path : C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset
Train Path   : C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset\train
Validation   : C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset\validation
Test Path    : C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\Traffic Violations Dataset\test


In [4]:
classes = sorted(
    [folder.name for folder in train_path.iterdir() if folder.is_dir()]
)

print("\nClasses:")
for i, class_name in enumerate(classes):
    print(f"{i}: {class_name}")

print("\nNumber of Classes:", len(classes))



Classes:
0: helmet
1: no_helmet
2: overloading

Number of Classes: 3


In [5]:

model = YOLO("yolo11n-cls.pt")

print("\nPretrained YOLO11 Classification Model Loaded")



Pretrained YOLO11 Classification Model Loaded


In [6]:
import time
import torch

# ============================================================
# START TIMER
# ============================================================

start_time = time.time()


# ============================================================
# CALLBACK - PRINT METRICS AFTER EACH EPOCH
# ============================================================

def print_epoch_metrics(trainer):

    epoch = trainer.epoch + 1

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{trainer.epochs}")
    print("=" * 60)

    # -----------------------------
    # Training Loss
    # -----------------------------

    if trainer.loss is not None:
        try:
            loss = float(trainer.loss)
            print(f"Training Loss   : {loss:.4f}")
        except:
            print(f"Training Loss   : {trainer.loss}")

    # -----------------------------
    # Accuracy
    # -----------------------------

    metrics = trainer.metrics

    if metrics:

        top1 = metrics.get(
            "metrics/accuracy_top1"
        )

        top5 = metrics.get(
            "metrics/accuracy_top5"
        )

        if top1 is not None:
            print(
                f"Top-1 Accuracy  : {top1 * 100:.2f}%"
            )

        if top5 is not None:
            print(
                f"Top-5 Accuracy  : {top5 * 100:.2f}%"
            )

    # -----------------------------
    # Epoch Time
    # -----------------------------

    current_time = time.time()
    elapsed = current_time - start_time

    minutes = int(elapsed // 60)
    seconds = int(elapsed % 60)

    print(
        f"Elapsed Time    : "
        f"{minutes}m {seconds}s"
    )

    print("=" * 60)


# Register callback
model.add_callback(
    "on_fit_epoch_end",
    print_epoch_metrics
)


# ============================================================
# TRAIN MODEL
# ============================================================

results = model.train(
    data=str(dataset_path),

    # Training settings
    epochs=30,
    imgsz=640,
    batch=16,

    # CPU / GPU
    device=0 if torch.cuda.is_available() else "cpu",

    # Data loading
    workers=4,

    # Output
    project="Helmet_Runs",
    name="helmet_classification",

    # Save model
    save=True,
    save_period=-1,

    # Validation
    val=True,

    # Display training information
    verbose=True
)


# ============================================================
# TOTAL TRAINING TIME
# ============================================================

end_time = time.time()

total_time = end_time - start_time

hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)
seconds = int(total_time % 60)


print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(
    f"Total Training Time : "
    f"{hours}h {minutes}m {seconds}s"
)

print(
    f"Total Time Seconds  : "
    f"{total_time:.2f}"
)

print("=" * 60)

New https://pypi.org/project/ultralytics/8.4.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.120  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Traffic Violations Dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, m

C:\Users\Nishant\AppData\Local\Temp\ipykernel_16940\4027341241.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  loss = float(trainer.loss)


       2/30      2.15G     0.4455          6        640: 100% ━━━━━━━━━━━━ 113/113 4.8it/s 23.8s0.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 12.5it/s 0.8s.1s
                   all       0.92          1

EPOCH 2/30
Training Loss   : 0.4800
Top-1 Accuracy  : 92.00%
Top-5 Accuracy  : 100.00%
Elapsed Time    : 1m 37s

      Epoch    GPU_mem       loss  Instances       Size
       3/30      2.15G     0.4204          6        640: 100% ━━━━━━━━━━━━ 113/113 4.0it/s 28.0s0.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 12.4it/s 0.8s2s
                   all      0.903          1

EPOCH 3/30
Training Loss   : 0.1238
Top-1 Accuracy  : 90.33%
Top-5 Accuracy  : 100.00%
Elapsed Time    : 2m 7s

      Epoch    GPU_mem       loss  Instances       Size
       4/30      2.15G     0.3952          6        640: 100% ━━━━━━━━━━━━ 113/113 4.0it/s 28.0s0.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 13.5it/s 0.7s2s
         

In [7]:
best_model_path = (
    Path(results.save_dir)
    / "weights"
    / "best.pt"
)

last_model_path = (
    Path(results.save_dir)
    / "weights"
    / "last.pt"
)


print("\n============================================")
print("TRAINING COMPLETED")
print("============================================")

print("\nBest Model:")
print(best_model_path)

print("\nLast Model:")
print(last_model_path)



TRAINING COMPLETED

Best Model:
C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\runs\classify\Helmet_Runs\helmet_classification\weights\best.pt

Last Model:
C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\runs\classify\Helmet_Runs\helmet_classification\weights\last.pt


In [8]:
model_folder = Path("saved_models")
model_folder.mkdir(exist_ok=True)

final_model_path = model_folder / "helmet_model.pt"

shutil.copy2(
    best_model_path,
    final_model_path
)

WindowsPath('saved_models/helmet_model.pt')

In [9]:
print("\n============================================")
print("MODEL SAVED SUCCESSFULLY")
print("============================================")

print(f"Model saved at:")
print(final_model_path.resolve())

print("\nYou can use this model later for testing:")
print("saved_models/helmet_model.pt")


MODEL SAVED SUCCESSFULLY
Model saved at:
C:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\saved_models\helmet_model.pt

You can use this model later for testing:
saved_models/helmet_model.pt
